In [3]:
from pathlib import Path
import json
import pandas as pd
from needle_bridge import NeedleBridgeConfig, build_needle_run_plan

# =========================================================
# PATHS / TOGGLES
# =========================================================

BASE_MATCHES_ROOT = Path("/media/hello/Vault/Tribunals/_Matches").resolve()
OUT_DIR = Path("/home/hello/Projects/Statements/output").resolve()
OUT_DIR.mkdir(parents=True, exist_ok=True)

BUILD_CSV_BRIDGE_FLOW = True
BUILD_MANUAL_Y_FLOW = True

# --- ET matches (mode-specific) ---
MATCHES_INDEX_BY_MODE = {
    "offensive": BASE_MATCHES_ROOT / "offensive" / "_matches_index.csv",
    "defensive": BASE_MATCHES_ROOT / "defensive" / "_matches_index.csv",
}

# --- WS enhanced (mode-specific: different has__ columns per mode) ---
WS_ENHANCED_BY_MODE = {
    "offensive": OUT_DIR / "Leonardo_WS_enhanced_offensive.csv",
    "defensive": OUT_DIR / "Leonardo_WS_enhanced_defensive.csv",
}

# --- Y sources ---
Y_INFERRED_JSON = OUT_DIR / "Y_inferred.json"
Y_MANUAL_JSON = OUT_DIR / "Y_manual.json"

# =========================================================
# HELPERS
# =========================================================

def _pack_tests(g):
    seen = set()
    out = []
    for _, r in g.iterrows():
        k = r["x_key"]
        if k in seen:
            continue
        seen.add(k)
        out.append({
            "x_key": k,
            "x_name": r["x_name"],
            "x_scope": r["x_scope"],
        })
    return out


def _build_grouped_jobs(run_plan_df):
    if run_plan_df.empty:
        return pd.DataFrame()

    return (
        run_plan_df
        .groupby(["row_id", "y_row_id", "et_path", "precedent_mode", "match_mode"], dropna=False)
        .apply(lambda g: pd.Series({
            "matched_needles": sorted(set(g["matched_needle"].dropna())),
            "x_tests": _pack_tests(g),
        }), include_groups=False)
        .reset_index()
    )


def _normalize_precedent_mode(value):
    s = str(value or "manual").strip().lower()
    return s if s else "manual"


def _extract_x_tests(y_obj):
    if not isinstance(y_obj, dict):
        return []

    raw = y_obj.get("x_tests") or {}
    out = []

    if isinstance(raw, dict):
        for x_key, x_obj in raw.items():
            x_obj = x_obj or {}
            out.append({
                "x_key": x_key,
                "x_name": x_obj.get("name", x_key),
                "x_scope": x_obj.get("scope", "GENERAL"),
            })
        return out

    if isinstance(raw, list):
        for item in raw:
            if not isinstance(item, dict):
                continue
            x_key = item.get("x_key") or item.get("key") or item.get("id")
            if not x_key:
                continue
            out.append({
                "x_key": x_key,
                "x_name": item.get("x_name") or item.get("name") or x_key,
                "x_scope": item.get("x_scope") or item.get("scope") or "GENERAL",
            })

    return out


def _iter_manual_y_records(path: Path):
    root = json.loads(path.read_text(encoding="utf-8"))

    if isinstance(root, dict) and isinstance(root.get("rows"), dict):
        for y_row_id, row_obj in root["rows"].items():
            row_obj = row_obj or {}
            yield y_row_id, row_obj
        return

    if isinstance(root, dict) and isinstance(root.get("items"), list):
        for i, row_obj in enumerate(root["items"], start=1):
            row_obj = row_obj or {}
            y_row_id = row_obj.get("y_row_id") or row_obj.get("id") or f"MANUAL_{i:04d}"
            yield y_row_id, row_obj
        return

    if isinstance(root, list):
        for i, row_obj in enumerate(root, start=1):
            row_obj = row_obj or {}
            y_row_id = row_obj.get("y_row_id") or row_obj.get("id") or f"MANUAL_{i:04d}"
            yield y_row_id, row_obj
        return

    raise ValueError(
        "Y_manual.json must be either {'rows': {...}}, {'items': [...]}, or a top-level list."
    )


def build_manual_y_parquets(y_manual_path: Path, out_dir: Path):
    if not y_manual_path.exists():
        raise FileNotFoundError(f"Missing manual Y source: {y_manual_path}")

    row_records = []
    x_records = []

    for y_row_id, row_obj in _iter_manual_y_records(y_manual_path):
        payload = row_obj.get("y") if isinstance(row_obj, dict) and isinstance(row_obj.get("y"), dict) else row_obj
        x_tests = _extract_x_tests(payload if isinstance(payload, dict) else {})

        row_id = row_obj.get("row_id") if isinstance(row_obj, dict) else None
        if row_id in (None, ""):
            row_id = y_row_id

        precedent_mode = _normalize_precedent_mode(
            row_obj.get("precedent_mode") if isinstance(row_obj, dict) else None
        )
        if precedent_mode == "manual":
            precedent_mode = _normalize_precedent_mode(
                row_obj.get("mode") if isinstance(row_obj, dict) else None
            )

        row_records.append({
            "row_id": row_id,
            "y_row_id": y_row_id,
            "precedent_mode": precedent_mode,
            "match_mode": "MANUAL_Y",
            "x_tests": x_tests,
            "x_test_count": len(x_tests),
            "source_json": str(y_manual_path),
        })

        for x in x_tests:
            x_records.append({
                "row_id": row_id,
                "y_row_id": y_row_id,
                "precedent_mode": precedent_mode,
                "match_mode": "MANUAL_Y",
                **x,
            })

    rows_df = pd.DataFrame(row_records)
    x_tests_df = pd.DataFrame(x_records)

    rows_out = out_dir / "grouped_manual_y_rows.parquet"
    x_tests_out = out_dir / "grouped_manual_y_x_tests.parquet"

    rows_df.to_parquet(rows_out, index=False)
    x_tests_df.to_parquet(x_tests_out, index=False)

    print("=" * 80)
    print("[manual_y] rows:", len(rows_df))
    print("[manual_y] x_tests:", len(x_tests_df))
    print(f"[manual_y] Saved -> {rows_out}")
    print(f"[manual_y] Saved -> {x_tests_out}")

    return rows_df, x_tests_df


# =========================================================
# FLOW 1: CSV -> Y_INFERRED -> GROUPED JOBS
# =========================================================

CSV_BRIDGE_OUT = {}
CSV_GROUPED = []

if BUILD_CSV_BRIDGE_FLOW:
    for mode in ["offensive", "defensive"]:
        print("=" * 80)
        print(f"[csv_bridge mode] {mode}")

        cfg = NeedleBridgeConfig(
            matches_index_csv=MATCHES_INDEX_BY_MODE[mode],
            match_frequencies_csv=None,
            ws_enhanced_csv=WS_ENHANCED_BY_MODE[mode],
            y_inferred_json=Y_INFERRED_JSON,
            et_path_col="path",
            ws_row_id_col="X1",
            out_row_id_col="row_id",
            y_row_prefix="X1_",
            y_row_pad=4,
            filter_ws_any_needle=True,
            filter_et_any_needle=True,
        )

        out = build_needle_run_plan(cfg)

        et_df = out["et_df"].copy()
        ws_df = out["ws_df"].copy()
        row_x_tests_df = out["row_x_tests_df"].copy()
        run_plan_df = out["run_plan_df"].copy()

        et_df["precedent_mode"] = mode
        ws_df["precedent_mode"] = mode
        row_x_tests_df["precedent_mode"] = mode
        run_plan_df["precedent_mode"] = mode

        grouped_jobs_df = _build_grouped_jobs(run_plan_df)
        grouped_jobs_df["precedent_mode"] = mode

        print("ET docs:", len(et_df))
        print("WS rows:", len(ws_df))
        print("Y rows:", len(row_x_tests_df))
        print("Run plan:", len(run_plan_df))
        print("Grouped jobs:", len(grouped_jobs_df))
        print()

        out_path = OUT_DIR / f"grouped_jobs_df_{mode}.parquet"
        grouped_jobs_df.to_parquet(out_path, index=False)
        print(f"Saved -> {out_path}")
        print()

        CSV_BRIDGE_OUT[mode] = {
            "grouped": grouped_jobs_df,
            "run_plan": run_plan_df,
        }
        CSV_GROUPED.append(grouped_jobs_df)

    combined_df = pd.concat(CSV_GROUPED, ignore_index=True)
    combined_path = OUT_DIR / "grouped_jobs_df_all_modes.parquet"
    combined_df.to_parquet(combined_path, index=False)

    print("=" * 80)
    print("[csv_bridge] Combined saved ->", combined_path)
    display(combined_df.head(20))

# =========================================================
# FLOW 2: Y_MANUAL -> MANUAL Y PARQUETS
# =========================================================

MANUAL_Y_ROWS_DF = None
MANUAL_Y_X_TESTS_DF = None

if BUILD_MANUAL_Y_FLOW:
    MANUAL_Y_ROWS_DF, MANUAL_Y_X_TESTS_DF = build_manual_y_parquets(
        y_manual_path=Y_MANUAL_JSON,
        out_dir=OUT_DIR,
    )

    if MANUAL_Y_ROWS_DF is not None and not MANUAL_Y_ROWS_DF.empty:
        display(MANUAL_Y_ROWS_DF.head(20))

    if MANUAL_Y_X_TESTS_DF is not None and not MANUAL_Y_X_TESTS_DF.empty:
        display(MANUAL_Y_X_TESTS_DF.head(20))


[csv_bridge mode] offensive
[bridge] using needle columns: ['has__any_needle', 'has__absence_of_contemporaneous_prohibition', 'has__appeal', 'has__disciplinary_hearing', 'has__duties', 'has__evidence', 'has__failure_to_engage_core_defence', 'has__grievance', 'has__gross_misconduct', 'has__instructions', 'has__investigation', 'has__logical_inconsistency_performance', 'has__management_acquiescence', 'has__no_contemporaneous_evidence', 'has__performance', 'has__post_hoc_intent_inference', 'has__predetermination', 'has__predetermination_and_appeal_failure', 'has__procedural_prejudice', 'has__role', 'has__undefined_primary_duty', 'has__warning']
ET docs: 12993
WS rows: 10
Y rows: 57
Run plan: 1931900
Grouped jobs: 115283

Saved -> /home/hello/Projects/Statements/output/grouped_jobs_df_offensive.parquet

[csv_bridge mode] defensive
[bridge] using needle columns: ['has__any_needle', 'has__band_of_reasonable_responses', 'has__credibility_inconsistency_attack', 'has__disciplinary_hearing', 'has

,row_id,y_row_id,et_path,precedent_mode,match_mode,matched_needles,x_tests
0,2,X1_0002,/media/hello/Vault/Tribunals/ET_Cases/1._Miss_...,offensive,NEEDLE_OVERLAP,"[has__duties, has__instructions, has__performa...","[{'x_key': 'X1', 'x_name': 'Management Role De..."
1,2,X1_0002,/media/hello/Vault/Tribunals/ET_Cases/1._Mr_An...,offensive,NEEDLE_OVERLAP,"[has__duties, has__instructions, has__performa...","[{'x_key': 'X1', 'x_name': 'Management Role De..."
2,2,X1_0002,/media/hello/Vault/Tribunals/ET_Cases/1._Mr_Ek...,offensive,NEEDLE_OVERLAP,"[has__duties, has__instructions, has__performa...","[{'x_key': 'X1', 'x_name': 'Management Role De..."
3,2,X1_0002,/media/hello/Vault/Tribunals/ET_Cases/1._Mr_G_...,offensive,NEEDLE_OVERLAP,"[has__instructions, has__management_acquiescen...","[{'x_key': 'X1', 'x_name': 'Management Role De..."
4,2,X1_0002,/media/hello/Vault/Tribunals/ET_Cases/1._Mr_M_...,offensive,NEEDLE_OVERLAP,"[has__absence_of_contemporaneous_prohibition, ...","[{'x_key': 'X1', 'x_name': 'Management Role De..."
5,2,X1_0002,/media/hello/Vault/Tribunals/ET_Cases/1._Mr_S_...,offensive,NEEDLE_OVERLAP,"[has__performance, has__role]","[{'x_key': 'X1', 'x_name': 'Management Role De..."
6,2,X1_0002,/media/hello/Vault/Tribunals/ET_Cases/13000462...,offensive,NEEDLE_OVERLAP,"[has__duties, has__instructions, has__performa...","[{'x_key': 'X1', 'x_name': 'Management Role De..."
7,2,X1_0002,/media/hello/Vault/Tribunals/ET_Cases/1300640_...,offensive,NEEDLE_OVERLAP,"[has__instructions, has__role]","[{'x_key': 'X1', 'x_name': 'Management Role De..."
8,2,X1_0002,/media/hello/Vault/Tribunals/ET_Cases/1301202_...,offensive,NEEDLE_OVERLAP,"[has__performance, has__role]","[{'x_key': 'X1', 'x_name': 'Management Role De..."
9,2,X1_0002,/media/hello/Vault/Tribunals/ET_Cases/1301253_...,offensive,NEEDLE_OVERLAP,"[has__duties, has__role]","[{'x_key': 'X1', 'x_name': 'Management Role De..."


FileNotFoundError: Missing manual Y source: /home/hello/Projects/Statements/output/Y_manual.json

In [ ]:
import sys, json, importlib, time
from pathlib import Path
from pprint import pprint
from tqdm.auto import tqdm

# ==========================================================
# MOLTIE RUNNER (GROUPED JOBS)
#  - consumes grouped_jobs_df (preferred) or run_plan_df (auto-groups)
#  - parses each PDF once per (row_id, et_path), runs all x_tests in-memory
# ==========================================================

# -------------------------
# REQUIRE: grouped_jobs_df OR run_plan_df exists
# -------------------------
if "grouped_jobs_df" in globals():
    jobs_df = grouped_jobs_df.copy()

elif "run_plan_df" in globals():
    # auto-group from run_plan_df
    required_cols = {"row_id", "y_row_id", "et_path", "x_key", "x_name", "x_scope", "matched_needle", "match_mode", "precedent_mode"}
    missing = required_cols - set(run_plan_df.columns)
    assert not missing, f"run_plan_df missing columns (for auto-group): {missing}"

    import pandas as pd

    def _pack_tests(g):
        seen = set()
        out = []
        for _, r in g.iterrows():
            k = r["x_key"]
            if k in seen:
                continue
            seen.add(k)
            out.append({"x_key": k, "x_name": r.get("x_name", k), "x_scope": r.get("x_scope", "GENERAL")})
        return out

    jobs_df = (
        run_plan_df
        .groupby(["row_id", "y_row_id", "et_path", "precedent_mode", "match_mode"], dropna=False)
        .apply(lambda g: pd.Series({
            "matched_needles": sorted(set(g["matched_needle"])),
            "x_tests": _pack_tests(g),
        }), include_groups=False)
        .reset_index()
    )
else:
    raise AssertionError("Missing grouped_jobs_df or run_plan_df. Build it with the Needle Bridge first.")

required_job_cols = {"row_id", "y_row_id", "et_path", "match_mode", "matched_needles", "x_tests"}
missing = required_job_cols - set(jobs_df.columns)
assert not missing, f"jobs_df missing columns: {missing}"

# -------------------------
# USER CONTROLS
# -------------------------
DEBUG = False   

# DEBUG selection: can be int, list[int], or slice
PLAN_ILOC = [3]  # examples: 3, [3], [2,5,9], slice(0,3)

# Force single PDF in DEBUG mode
FORCE_PDF = Path("/home/hello/Projects/Statements/code/appeals/1._Mr_G_Lepiarz__2._Mr_D_Lewis_v__Trades_Union_Congress_-_2200228-2023___2200230-2023.pdf")

# Batch caps (only used when DEBUG=False)
ONLY_X_KEY = None              # e.g. "X3" -> keep only that test inside x_tests list
MAX_JOBS = None                # e.g. 2000 -> cap number of PDFs/jobs (recommended)
# (Your old MAX_DOCS_PER_YROW_AND_X is less meaningful now; grouping changed the unit of work.)

# Output
REPO_ROOT = Path("/home/hello/Projects/Statements").resolve()
OUT_DIR = REPO_ROOT / "output" / "moltie_batch"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# -------------------------
# Repo / code path / Y path
# -------------------------
CODE_ROOT = (REPO_ROOT / "code").resolve()
Y_PATH = REPO_ROOT / "output" / "Y_inferred.json"

assert CODE_ROOT.exists(), f"Missing CODE_ROOT: {CODE_ROOT}"
assert Y_PATH.exists(), f"Missing Y: {Y_PATH}"

if str(CODE_ROOT) not in sys.path:
    sys.path.insert(0, str(CODE_ROOT))

# -------------------------
# Imports (reload once)
# -------------------------
import moltie.schemas.run_config as rc_mod
import moltie.llm.verifier_prompt as vp_mod
import moltie.llm.client as client_mod
import moltie.schemas.query_object as qo_mod
import moltie.agent.loop as loop_mod

importlib.reload(rc_mod)
importlib.reload(vp_mod)
importlib.reload(client_mod)
importlib.reload(qo_mod)
importlib.reload(loop_mod)

RunConfig = rc_mod.RunConfig
AtomQuery = qo_mod.AtomQuery
run_agent_on_one_doc = loop_mod.run_agent_on_one_doc
LLMClientConfig = client_mod.LLMClientConfig

# -------------------------
# Load Y once
# -------------------------
y_root = json.loads(Y_PATH.read_text(encoding="utf-8"))
rows = y_root.get("rows") or {}
assert isinstance(rows, dict) and rows, "Y has no rows"

# -------------------------
# Build plan_jobs (DEBUG override vs batch)
# -------------------------
if DEBUG:
    assert FORCE_PDF.exists(), f"Forced PDF missing: {FORCE_PDF}"

    if isinstance(PLAN_ILOC, slice):
        plan_jobs = jobs_df.iloc[PLAN_ILOC].copy()
    elif isinstance(PLAN_ILOC, list):
        plan_jobs = jobs_df.iloc[PLAN_ILOC].copy()
    elif isinstance(PLAN_ILOC, int):
        plan_jobs = jobs_df.iloc[[PLAN_ILOC]].copy()
    else:
        raise ValueError("PLAN_ILOC must be int, list, or slice")

    plan_jobs["et_path"] = str(FORCE_PDF)
else:
    plan_jobs = jobs_df.copy()

    if ONLY_X_KEY is not None:
        def _filter_tests(lst):
            out = [t for t in (lst or []) if t.get("x_key") == ONLY_X_KEY]
            return out
        plan_jobs["x_tests"] = plan_jobs["x_tests"].apply(_filter_tests)
        plan_jobs = plan_jobs[plan_jobs["x_tests"].apply(lambda x: len(x or []) > 0)].copy()

    if MAX_JOBS is not None:
        plan_jobs = plan_jobs.head(int(MAX_JOBS)).copy()

plan_jobs = plan_jobs.reset_index(drop=True)

print("DEBUG:", DEBUG)
print("Jobs:", len(plan_jobs))
print("Unique y_row_id:", plan_jobs["y_row_id"].nunique(), "| unique PDFs:", plan_jobs["et_path"].nunique())
print(plan_jobs[["row_id", "y_row_id", "match_mode", "et_path"]].head(10).to_string(index=False))

# -------------------------
# Client cfg (pinned once)
# -------------------------
_client_kwargs = dict(
    model="mistral-small3.2:latest",
    ollama_url="http://localhost:11434/api/generate",
    timeout_s=180,
    temperature=0.0,
    num_predict=1000,
    max_retries=2,
)

# optional stop/debug fields
try:
    if "stop" in getattr(LLMClientConfig, "__annotations__", {}):
        _client_kwargs["stop"] = []
except Exception:
    pass

try:
    if "debug" in getattr(LLMClientConfig, "__annotations__", {}):
        _client_kwargs["debug"] = DEBUG
except Exception:
    pass

client_cfg = LLMClientConfig(**_client_kwargs)

# -------------------------
# RunConfig (pinned once)
# -------------------------
cfg2 = RunConfig.from_dict({
    "debug": DEBUG,
    "harvest_mode": False,
    "max_iters": 2,
    "window_size": 12,
    "stride": 12,
    "top_windows": 1,
    "k_chunks_per_doc": 4,
    "anchors_required": 1,
    "min_hits": 1,
    "thresh_score": 1,
    "thresh_conf": 0.8,
    "plateau_p": 1,
    "eps_improve": 0,
    "iter_temp_enabled": False,
})

# -------------------------
# PDF -> paras cache
# -------------------------
paras_cache = {}  # pdf_path_str -> paras(list[dict])

def get_paras(pdf_path: Path):
    k = str(pdf_path)
    if k in paras_cache:
        return paras_cache[k]

    from pypdf import PdfReader
    reader = PdfReader(str(pdf_path))
    text = "\n".join([(p.extract_text() or "") for p in reader.pages]).strip()
    paras = [{"para_id": "p00001", "text": text}]
    paras = loop_mod._maybe_rechunk_single_blob_paras(paras)

    paras_cache[k] = paras
    return paras

# -------------------------
# Output JSONL
# -------------------------
ts = time.strftime("%Y%m%d_%H%M%S")
OUT_JSONL = OUT_DIR / f"batch_results_{'DEBUG' if DEBUG else 'BATCH'}_{ts}.jsonl"
print("\nWriting results to:", OUT_JSONL)

def safe_to_dict(obj):
    if obj is None:
        return None
    if hasattr(obj, "to_dict"):
        return obj.to_dict()
    if isinstance(obj, dict):
        return obj
    return {"repr": repr(obj)}

n_ok = 0
n_neg = 0
n_err = 0

with OUT_JSONL.open("w", encoding="utf-8") as f:
    for j, job in tqdm(plan_jobs.iterrows(), total=len(plan_jobs), desc="moltie jobs", unit="job"):
        y_row_id = job["y_row_id"]
        pdf_path = Path(job["et_path"])
        row_id = job.get("row_id")
        precedent_mode = job.get("precedent_mode")
        match_mode = job.get("match_mode")
        matched_needles = job.get("matched_needles") or []

        try:
            if y_row_id not in rows:
                raise KeyError(f"y_row_id not in Y.rows: {y_row_id}")
            if not pdf_path.exists():
                raise FileNotFoundError(f"PDF missing: {pdf_path}")

            # row-scoped y object
            y_obj = (rows[y_row_id] or {}).get("y") or {}

            # parse / rechunk ONCE per PDF
            paras = get_paras(pdf_path)
            doc_id = pdf_path.stem

            # run all x_tests for this job
            for t in (job.get("x_tests") or []):
                x_key = t.get("x_key")
                x_name = t.get("x_name", x_key)

                try:
                    merged = qo_mod.merge_indicators_and_excludes(y_obj, [x_key])
                    atom = AtomQuery(
                        atom_id=x_key,
                        x_tests=[x_key],
                        proposition=x_name,
                        positive_indicators=merged["positive_indicators"],
                        excludes=merged["excludes"],
                        keyword_seeds=merged["positive_indicators"],
                        expansion_terms=[],
                    )

                    res = run_agent_on_one_doc(doc_id, paras, atom, cfg2, client_cfg)

                    verdict = safe_to_dict(getattr(res, "verdict", None))
                    negative_exit = safe_to_dict(getattr(res, "negative_exit", None))

                    if verdict:
                        n_ok += 1
                    else:
                        n_neg += 1

                    row_out = {
                        "job_i": int(j),
                        "precedent_mode": precedent_mode,
                        "row_id": row_id,
                        "y_row_id": y_row_id,
                        "match_mode": match_mode,
                        "matched_needles": matched_needles,
                        "et_path": str(pdf_path),
                        "doc_id": doc_id,
                        "x_key": x_key,
                        "x_name": x_name,
                        "verdict": verdict,
                        "negative_exit": negative_exit,
                        "iters": getattr(res, "iters", None),
                        "trace_tail": (getattr(res, "trace", None) or [])[-3:],
                    }
                    f.write(json.dumps(row_out, ensure_ascii=False) + "\n")

                    if DEBUG:
                        print("\n--- JOB", j, "| X", x_key, "---")
                        print("y_row_id:", y_row_id, "| pdf:", pdf_path.name)
                        if verdict:
                            print("relevant=", verdict.get("relevant"),
                                  "score=", verdict.get("precedent_score"),
                                  "conf=", verdict.get("confidence"),
                                  "anchors=", len(verdict.get("anchors") or []))
                        else:
                            print("NEGATIVE:", (negative_exit or {}).get("reason"))

                except Exception as e_x:
                    n_err += 1
                    f.write(json.dumps({
                        "job_i": int(j),
                        "precedent_mode": precedent_mode,
                        "row_id": row_id,
                        "y_row_id": y_row_id,
                        "match_mode": match_mode,
                        "et_path": str(pdf_path),
                        "doc_id": doc_id,
                        "x_key": x_key,
                        "error": repr(e_x),
                    }, ensure_ascii=False) + "\n")

        except Exception as e_job:
            n_err += 1
            f.write(json.dumps({
                "job_i": int(j),
                "precedent_mode": precedent_mode,
                "row_id": row_id,
                "y_row_id": y_row_id,
                "match_mode": match_mode,
                "et_path": str(pdf_path),
                "error": repr(e_job),
            }, ensure_ascii=False) + "\n")

print("\nDONE")
print("ok:", n_ok, "| negative:", n_neg, "| errors:", n_err)
print("results:", OUT_JSONL)

DEBUG: True
Jobs: 1
Unique y_row_id: 1 | unique PDFs: 1
 row_id y_row_id     match_mode                                                                                                                                et_path
      2  X1_0002 NEEDLE_OVERLAP /home/hello/Projects/Statements/code/appeals/1._Mr_G_Lepiarz__2._Mr_D_Lewis_v__Trades_Union_Congress_-_2200228-2023___2200230-2023.pdf

Writing results to: /home/hello/Projects/Statements/output/moltie_batch/batch_results_DEBUG_20260423_170640.jsonl


moltie jobs:   0%|          | 0/1 [00:00<?, ?job/s]

[moltie.loop] start doc_id='1._Mr_G_Lepiarz__2._Mr_D_Lewis_v__Trades_Union_Congress_-_2200228-2023___2200230-2023' atom_id='X1' n_paras=84

[moltie.client] ===== Attempt A =====
[moltie.client] attempt: 0
[moltie.client] prompt_hash: c5648b22bc
[moltie.client] raw_len: 331
[moltie.client] raw_head:
 {
  "relevant": false,
  "precedent_score": 0,
  "confidence": 0,
  "anchors": [],
  "use_mode": "contrast",
  "proposition_winner": "unclear",
  "appeal_outcome": "unknown",
  "successful_party": "unclear",
  "distinguishers": [],
  "note": "No relevant information found.",
  "retrieval_score": null,
  "retrieval_method": null
}
[moltie.client] raw_tail:
 {
  "relevant": false,
  "precedent_score": 0,
  "confidence": 0,
  "anchors": [],
  "use_mode": "contrast",
  "proposition_winner": "unclear",
  "appeal_outcome": "unknown",
  "successful_party": "unclear",
  "distinguishers": [],
  "note": "No relevant information found.",
  "retrieval_score": null,
  "retrieval_method": null
}
[moltie.

In [ ]:
from pathlib import Path
import json
import pandas as pd

# =========================================================
# MASTER MOLTIE -> CSV PIPELINE
# 1) Load MASTER_moltie.jsonl
# 2) Promote negative_exit.best_attempt when verdict is null
# 3) Flatten key verdict fields
# 4) Split into relevant / remaining
# 5) Sort relevant by (anchor_count desc, confidence desc)
# 6) Save:
#    - MASTER_moltie.csv (all rows flattened)
#    - MASTER_moltie__relevant.csv
#    - MASTER_moltie__remaining.csv
# 7) Collapse by et_path (one row per PDF), compress x_key/x_name with "||"
#    - MASTER_moltie__relevant__by_pdf.csv
#    - MASTER_moltie__remaining__by_pdf.csv
# =========================================================

# -------------------------
# INPUT
# -------------------------
JSONL_PATH = Path("/home/hello/Projects/Statements/output/moltie_batch/MASTER_moltie.jsonl")
print("Loading:", JSONL_PATH)
assert JSONL_PATH.exists(), f"Missing: {JSONL_PATH}"

# -------------------------
# LOAD JSONL
# -------------------------
rows = []
with JSONL_PATH.open("r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        rows.append(json.loads(line))

df = pd.DataFrame(rows)
print("Raw shape:", df.shape)

# -------------------------
# PROMOTE best_attempt IF verdict is None
# -------------------------
def effective_verdict(row):
    v = row.get("verdict")
    if isinstance(v, dict):
        return v
    neg = row.get("negative_exit")
    if isinstance(neg, dict):
        best = neg.get("best_attempt")
        if isinstance(best, dict):
            return best
    return None

df["effective_verdict"] = df.apply(effective_verdict, axis=1)

# -------------------------
# FLATTEN verdict fields
# -------------------------
def extract_field(d, key):
    if isinstance(d, dict):
        return d.get(key)
    return None

df["relevant"] = df["effective_verdict"].apply(lambda v: extract_field(v, "relevant"))
df["precedent_score"] = df["effective_verdict"].apply(lambda v: extract_field(v, "precedent_score"))
df["confidence"] = df["effective_verdict"].apply(lambda v: extract_field(v, "confidence"))

df["anchor_count"] = df["effective_verdict"].apply(
    lambda v: len(v.get("anchors") or []) if isinstance(v, dict) else 0
)

df["negative_reason"] = (
    df["negative_exit"].apply(lambda v: v.get("reason") if isinstance(v, dict) else None)
    if "negative_exit" in df.columns
    else pd.Series([None] * len(df), dtype=object)
)

# -------------------------
# SELECT CLEAN COLUMNS
# -------------------------
final_cols = [
    "row_id",
    "y_row_id",
    "x_key",
    "x_name",
    "et_path",
    "relevant",
    "precedent_score",
    "confidence",
    "anchor_count",
    "negative_reason",
]

# Keep only columns that exist (defensive)
final_cols = [c for c in final_cols if c in df.columns]

df_out = df[final_cols].copy()
print("Flattened shape:", df_out.shape)

# -------------------------
# SPLIT: relevant vs remaining
# -------------------------
df_relevant = df_out[df_out["relevant"] == True].copy()
df_remaining = df_out[df_out["relevant"] != True].copy()

# Sort relevant: strongest anchors first, then confidence
df_relevant = df_relevant.sort_values(
    by=["anchor_count", "confidence"],
    ascending=[False, False],
    na_position="last"
)

print("Relevant shape:", df_relevant.shape)
print("Remaining shape:", df_remaining.shape)

# -------------------------
# SAVE CSVs (row-level)
# -------------------------
BASE_NAME = JSONL_PATH.with_suffix("")  # Path without ".jsonl"

CSV_ALL = BASE_NAME.with_suffix(".csv")
CSV_RELEVANT = BASE_NAME.with_name(BASE_NAME.name + "__relevant.csv")
CSV_REMAINING = BASE_NAME.with_name(BASE_NAME.name + "__remaining.csv")

df_out.to_csv(CSV_ALL, index=False)
df_relevant.to_csv(CSV_RELEVANT, index=False)
df_remaining.to_csv(CSV_REMAINING, index=False)

print("Saved ALL:", CSV_ALL)
print("Saved RELEVANT:", CSV_RELEVANT)
print("Saved REMAINING:", CSV_REMAINING)

# -------------------------
# DEDUPE BY et_path (one row per PDF)
# compress Xs using "||"
# -------------------------
SEP = "||"

def _compress_unique_in_order(vals):
    out = []
    seen = set()
    for v in vals:
        if v is None:
            continue
        v = str(v)
        if v in seen:
            continue
        seen.add(v)
        out.append(v)
    return SEP.join(out)

def collapse_by_pdf(df_in: pd.DataFrame) -> pd.DataFrame:
    if df_in.empty:
        return pd.DataFrame(columns=[
            "et_path",
            "n_atoms",
            "max_anchor_count",
            "max_confidence",
            "max_precedent_score",
            "x_keys_comp",
            "x_names_comp",
            "best_row_id",
            "best_y_row_id",
        ])

    # Ensure sorted so "best" Xs appear first in the compression
    df_sorted = df_in.sort_values(
        by=["et_path", "anchor_count", "confidence"],
        ascending=[True, False, False],
        na_position="last"
    )

    rows_out = []
    for et_path, g in df_sorted.groupby("et_path", dropna=False):
        # g is already sorted best-first for this et_path
        rows_out.append({
            "et_path": et_path,
            "n_atoms": int(len(g)),
            "max_anchor_count": int(g["anchor_count"].fillna(0).max()),
            "max_confidence": float(g["confidence"].fillna(0).max()),
            "max_precedent_score": float(g["precedent_score"].fillna(0).max()) if "precedent_score" in g.columns else 0.0,
            "x_keys_comp": _compress_unique_in_order(g["x_key"].tolist()) if "x_key" in g.columns else "",
            "x_names_comp": _compress_unique_in_order(g["x_name"].tolist()) if "x_name" in g.columns else "",
            "best_row_id": g.iloc[0]["row_id"] if "row_id" in g.columns else None,
            "best_y_row_id": g.iloc[0]["y_row_id"] if "y_row_id" in g.columns else None,
        })

    out = pd.DataFrame(rows_out)

    # Sort PDFs by strongest evidence
    out = out.sort_values(
        by=["max_anchor_count", "max_confidence", "n_atoms"],
        ascending=[False, False, False],
        na_position="last"
    ).reset_index(drop=True)

    return out

df_relevant_pdf = collapse_by_pdf(df_relevant)
df_remaining_pdf = collapse_by_pdf(df_remaining)

# -------------------------
# SAVE CSVs (pdf-level)
# -------------------------
CSV_RELEVANT_PDF = BASE_NAME.with_name(BASE_NAME.name + "__relevant__by_pdf.csv")
CSV_REMAINING_PDF = BASE_NAME.with_name(BASE_NAME.name + "__remaining__by_pdf.csv")

df_relevant_pdf.to_csv(CSV_RELEVANT_PDF, index=False)
df_remaining_pdf.to_csv(CSV_REMAINING_PDF, index=False)

print("Saved RELEVANT by PDF:", CSV_RELEVANT_PDF, "| shape:", df_relevant_pdf.shape)
print("Saved REMAINING by PDF:", CSV_REMAINING_PDF, "| shape:", df_remaining_pdf.shape)

# -------------------------
# Preview
# -------------------------
print("\nTop relevant rows:")
display(df_relevant.head(10))

print("\nTop relevant PDFs:")
display(df_relevant_pdf.head(10))

In [ ]:
import pandas as pd
import json
from pathlib import Path

REPO_ROOT = Path("/home/hello/Projects/Statements").resolve()

GROUPED_PATH = REPO_ROOT / "output" / "grouped_jobs_df_all_modes.parquet"
Y_PATH = REPO_ROOT / "output" / "Y_inferred.json"
WS_PATH = REPO_ROOT / "input" / "Leonardo_WS.csv"

df = pd.read_parquet(GROUPED_PATH)
y = json.loads(Y_PATH.read_text())
ws = pd.read_csv(WS_PATH)

def as_list(x):
    if x is None:
        return []
    try:
        if pd.isna(x):
            return []
    except Exception:
        pass
    if hasattr(x, "tolist"):
        try:
            return x.tolist()
        except Exception:
            pass
    if isinstance(x, (list, tuple, set)):
        return list(x)
    return [x]

# -------------------------
# Correct counts
# -------------------------
df["n_x_tests"] = df["x_tests"].apply(lambda x: len(as_list(x)))
df["n_needles"] = df["matched_needles"].apply(lambda x: len(as_list(x)))

print("\n=== CORRECTED X TESTS STATS ===")
print(df["n_x_tests"].describe())

print("\n=== CORRECTED NEEDLES STATS ===")
print(df["n_needles"].describe())

# -------------------------
# Flatten x_tests
# -------------------------
flat_rows = []
for _, r in df.iterrows():
    for xt in as_list(r["x_tests"]):
        flat_rows.append({
            "row_id": r["row_id"],
            "y_row_id": r["y_row_id"],
            "et_path": r["et_path"],
            "precedent_mode": r["precedent_mode"],
            "match_mode": r["match_mode"],
            "x_key": xt.get("x_key"),
            "x_name": xt.get("x_name"),
            "x_scope": xt.get("x_scope"),
        })

flat_x = pd.DataFrame(flat_rows)

print("\n=== FLAT X SHAPE ===")
print(flat_x.shape)

print("\n=== ROWS PER X_KEY ===")
print(flat_x["x_key"].value_counts().sort_index())

print("\n=== UNIQUE PDFS PER X_KEY ===")
print(flat_x.groupby("x_key")["et_path"].nunique().sort_index())

print("\n=== ROWS PER ROW_ID ===")
print(df["row_id"].value_counts().sort_index())

print("\n=== ATOMS PER ROW_ID ===")
print(flat_x.groupby("row_id").size().sort_index())

print("\n=== PDFS PER ROW_ID ===")
print(df.groupby("row_id")["et_path"].nunique().sort_index())

print("\n=== X_KEY BY PRECEDENT MODE ===")
print(pd.crosstab(flat_x["x_key"], flat_x["precedent_mode"]))

print("\n=== X_KEY BY MATCH MODE ===")
print(pd.crosstab(flat_x["x_key"], flat_x["match_mode"]))

print("\n=== ROW_ID x X_KEY MATRIX ===")
row_x = pd.crosstab(flat_x["row_id"], flat_x["x_key"])
print(row_x)

print("\n=== AVG X TESTS PER JOB BY MATCH MODE ===")
print(df.groupby("match_mode")["n_x_tests"].mean())

print("\n=== AVG NEEDLES PER JOB BY MATCH MODE ===")
print(df.groupby("match_mode")["n_needles"].mean())

In [ ]:
# -------------------------
# X summary with sample PDFs
# -------------------------
import random

random.seed(42)

def sample_pdf_names(paths, k=5):
    vals = pd.Series(paths).dropna().astype(str).unique().tolist()
    vals = [Path(p).name for p in vals]
    if not vals:
        return []
    if len(vals) <= k:
        return sorted(vals)
    return sorted(random.sample(vals, k))

x_summary = (
    flat_x.groupby("x_key")
    .agg(
        x_name=("x_name", lambda s: s.dropna().iloc[0] if len(s.dropna()) else None),
        n_rows=("x_key", "size"),
        n_unique_pdfs=("et_path", "nunique"),
        sample_pdfs=("et_path", lambda s: sample_pdf_names(s, k=5)),
    )
    .reset_index()
    .sort_values("x_key")
)

print("\n=== X SUMMARY WITH SAMPLE PDFS ===")
for _, r in x_summary.iterrows():
    print("\n---")
    print("x_key:", r["x_key"])
    print("x_name:", r["x_name"])
    print("n_rows:", r["n_rows"])
    print("n_unique_pdfs:", r["n_unique_pdfs"])
    print("sample_pdfs:")
    for p in r["sample_pdfs"]:
        print("  -", p)